In [ ]:
# CELL 1 — Load the saved model (run once; no training).
# Local: keep this notebook inside the supplied evaluation folder.
# Colab: when prompted, upload dermasense_assets.zip from that folder.
from pathlib import Path
import hashlib, importlib.util, subprocess, sys, zipfile

packages = {"torch":"torch", "torchvision":"torchvision", "numpy":"numpy",
            "PIL":"Pillow", "sklearn":"scikit-learn", "pandas":"pandas",
            "matplotlib":"matplotlib", "ipywidgets":"ipywidgets"}
missing = [p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
assets = Path("_assets")
if not (assets / "dermasense_demo.py").exists():
    bundle = Path("dermasense_assets.zip")
    if not bundle.exists():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError("Open this notebook from the supplied evaluation folder, beside dermasense_assets.zip.")
        print("Upload dermasense_assets.zip once. Upload test images in cell 2.")
        uploaded = files.upload()
        matches = [data for name,data in uploaded.items() if name.lower().endswith(".zip")]
        if len(matches) != 1:
            raise ValueError("Select exactly one file: dermasense_assets.zip")
        bundle.write_bytes(matches[0])
    if hashlib.sha256(bundle.read_bytes()).hexdigest() != "b02314c9234061b4ee7847b5b50702d809afdd3072a6ffa26875a965b1e41057":
        raise ValueError("Wrong or incomplete assets ZIP. Use the ZIP supplied with this notebook.")
    allowed = {"dermasense_demo.py", "demo_model.pt", "demo_records.json", "asset_manifest.json"}
    assets.mkdir(exist_ok=True)
    with zipfile.ZipFile(bundle) as archive:
        if set(archive.namelist()) != allowed:
            raise ValueError("Unexpected asset contents")
        for name in allowed:
            (assets / name).write_bytes(archive.read(name))
spec = importlib.util.spec_from_file_location("dermasense_live_runtime", assets / "dermasense_demo.py")
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
demo = runtime.load_demo(assets)


In [ ]:
# CELL 2 — Upload any number of the 20 supplied images.
# Rerun this cell to replace your selection.
selection = runtime.pick_images()


In [ ]:
# CELL 3 — Predict and calculate metrics for the selected images.
# Repeat cells 2 and 3 for another live test; the model remains loaded.
results = runtime.run_demo(demo, selection)
